# Versionnage et traçage des données

Voyons comment utiliser et tracer des données au cours d'un projet de data science.

De nombreux outils permettent de le faire. Nous utiliserons [DVC](https://dvc.org/) pour cette démonstration.

In [1]:
# @title ## Configuration & installations
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!apt install tree
!pip install dvc




tree is already the newest version (2.1.1-2ubuntu3.24.04.2).
0 upgraded, 0 newly installed, 0 to remove and 146 not upgraded.


## Initialisation du dépôt du projet

In [2]:
!git init
!dvc init

Initialized empty Git repository in /content/.git/
Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


## Récupération des données

Il y a plusieurs manières d'importer des données dans le dépôt du projet. Nous utiliserons ici [`dvc get`](https://dvc.org/doc/command-reference/get), qui récupère des données depuis un dépôt git/DVC, puis nous les ferons suivre par DVC avec [`dvc add`](https://dvc.org/doc/command-reference/add).

In [3]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [4]:
!dvc add data.csv

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data.csv to cache:   0% 0/1 [00:00<?, ?file/s]
Adding data.csv to cache:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
                                                                 
Checking out /content/data.csv:   0% 0/1 [00:00<?, ?files/s]
Checking out /content/data.csv:   0% 0/1 [00:00<?, ?files/s{'info': ''}]
Adding...: 100% 1/1 [00:00<00:00,  4.09file/s{'info': ''}]

To track the changes with git, run:

	git add .gitignore data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [5]:
!cat data.csv.dvc
!tree .dvc

outs:
- md5: a546111eeba6497aafc1b3263cf45d6c
  size: 81193310
  hash: md5
  path: data.csv
.dvc
├── cache
│   └── files
│       └── md5
│           └── a5
│               └── 46111eeba6497aafc1b3263cf45d6c
├── config
└── tmp
    ├── btime
    ├── lock
    ├── rwlock
    └── rwlock.lock

6 directories, 6 files


## Enregistrement des métadonnées dans le dépôt du projet

In [6]:
!git add .gitignore data.csv.dvc

In [7]:
!git commit -m "Ajout d'une première version des données"
!git tag "v1"

[main (root-commit) 9211d49] Ajout d'une première version des données
 5 files changed, 12 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
 create mode 100644 .gitignore
 create mode 100644 data.csv.dvc


## Stockage des données sur un serveur

Avec DVC, les données sont stockées à part des métadonnées. Nous configurons ici un faux serveur de stockage sur cette machine, mais ce serait d'habitude un serveur dédié (sur AWS par exemple).

In [8]:
!dvc remote add -d local /tmp/dvcstore

Setting 'local' as a default remote.


In [9]:
!dvc push

Pushing
!
  0% |          |0/? [00:00<?,    ?files/s]
Pushing
Everything is up to date.


## Modification des données et suivi des changements

In [10]:
!sort -r < data.csv > a && dvc remove data.csv.dvc && mv a data.csv

In [11]:
!dvc add data.csv

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data.csv to cache:   0% 0/1 [00:00<?, ?file/s]
Adding data.csv to cache:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
                                                                 
Checking out /content/data.csv:   0% 0/1 [00:00<?, ?files/s]
Checking out /content/data.csv:   0% 0/1 [00:00<?, ?files/s{'info': ''}]
Adding...: 100% 1/1 [00:00<00:00,  4.27file/s{'info': ''}]

To track the changes with git, run:

	git add data.csv.dvc .gitignore

To enable auto staging, run:

	dvc config core.autostage true


In [12]:
!cat data.csv.dvc
!tree .dvc

outs:
- md5: 3b01e00ab894980e5b6b851be9292783
  size: 81193310
  hash: md5
  path: data.csv
.dvc
├── cache
│   └── files
│       └── md5
│           ├── 3b
│           │   └── 01e00ab894980e5b6b851be9292783
│           └── a5
│               └── 46111eeba6497aafc1b3263cf45d6c
├── config
└── tmp
    ├── btime
    ├── lock
    ├── rwlock
    └── rwlock.lock

7 directories, 7 files


In [13]:
!git add data.csv.dvc .gitignore

In [14]:
!git commit -m "Données v2"
!git tag "v2"

[main ef10639] Données v2
 1 file changed, 1 insertion(+), 1 deletion(-)


In [15]:
!dvc push

Pushing
!
  0% |          |0/? [00:00<?,    ?files/s]
Pushing
Everything is up to date.


## Exemple d'utilisation des versions : retour en arrière

Imaginons que la nouvelle version des données pose problème. On peut revenir facilement à la première version&nbsp;: c'est tout l'intérêt de versionner ses données.

In [16]:
!git checkout v1 data.csv.dvc

Updated 1 path from 9b1128e


In [17]:
!cat data.csv.dvc
!tree .dvc

outs:
- md5: a546111eeba6497aafc1b3263cf45d6c
  size: 81193310
  hash: md5
  path: data.csv
.dvc
├── cache
│   └── files
│       └── md5
│           ├── 3b
│           │   └── 01e00ab894980e5b6b851be9292783
│           └── a5
│               └── 46111eeba6497aafc1b3263cf45d6c
├── config
└── tmp
    ├── btime
    ├── lock
    ├── rwlock
    └── rwlock.lock

7 directories, 7 files


In [18]:
!dvc checkout data.csv

Building workspace index          |1.00 [00:00, 80.9entry/s]
Comparing indexes          |2.00 [00:00,  701entry/s]
Applying changes          |1.00 [00:00,   727file/s]
M       data.csv


On enregistre maintenant dans le dépôt les métadonnées indiquant ce retour en arrière.

In [19]:
!git add data.csv.dvc

In [20]:
!git commit -m "Retour aux données v1"
!git tag v3

[main 5f81851] Retour aux données v1
 1 file changed, 1 insertion(+), 1 deletion(-)


In [21]:
!dvc push

Pushing
!
  0% |          |0/? [00:00<?,    ?files/s]
Pushing
Everything is up to date.


## Traitement des données

In [22]:
%%writefile upper.py
from pathlib import Path
from sys import argv

Path(argv[2]).write_text(
    Path(argv[1]).read_text(encoding="utf8").upper(),
    encoding="utf8")

Writing upper.py


[`dvc stage add`](https://dvc.org/doc/command-reference/stage/add) ajoute une étape de traitement à nos métadonnées. On saura ainsi exactement quels traitements ont été appliqués et quels résultats ils ont produits.

In [23]:
!dvc stage add -n transform-uppercase -d data.csv -o data-upper.csv python upper.py data.csv data-upper.csv

Added stage 'transform-uppercase' in 'dvc.yaml'

To track the changes with git, run:

	git add .gitignore dvc.yaml

To enable auto staging, run:

	dvc config core.autostage true


[`dvc repro`](https://dvc.org/doc/command-reference/repro) exécute cette étape.

In [24]:
!dvc repro

'data.csv.dvc' didn't change, skipping
Running stage 'transform-uppercase':
> python upper.py data.csv data-upper.csv
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [25]:
!git add dvc.yaml .gitignore dvc.lock

In [26]:
!git commit -m "Pipeline pour passer un fichier en majuscule"

[main af6429c] Pipeline pour passer un fichier en majuscule
 3 files changed, 22 insertions(+)
 create mode 100644 dvc.lock
 create mode 100644 dvc.yaml


In [27]:
!dvc push

Pushing
!
  0% |          |0/? [00:00<?,    ?files/s]
Pushing
Everything is up to date.
